# PART IV: Clustering

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import itertools
import numpy as np
import pandas as pd
import umap
import seaborn as sns

from sklearn.base import clone
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics.cluster import contingency_matrix
from sklearn.neighbors import kneighbors_graph
from sklearn import metrics

## Importing data

In [ ]:
df = pd.read_parquet("advanced_feature_selected_data.parquet")
df.info()

## Data Pre-processing

Scale all continuous attributes to a 0 to 1 scale using MinMaxScaler

In [ ]:
continuous_columns = [
    "Start_Lat",
    "Start_Lng",
    "Distance(mi)",
    "Temperature(F)",
    "Wind_Chill(F)",
    "Humidity(%)",
    "Precipitation(in)",
    "Duration_Minutes",
]

scaler = MinMaxScaler().fit(df[continuous_columns])

Take 1% subset

In [ ]:
sample_df, _ = train_test_split(
    df,
    train_size=0.01,
    stratify=df["Severity"],
    random_state=42,
)

sample_df[continuous_columns] = scaler.transform(sample_df[continuous_columns])

X = sample_df.drop(columns=["Severity"])
y = sample_df["Severity"]

Drop boolean values

In [ ]:
X = X.drop(columns=X.select_dtypes(include=["bool", "boolean"]).columns)
X = X.astype("float32")

## Clustering

In [ ]:
def make_cluster_plot(model, embedding, ax, title=None):
    cluster_labels = np.asarray(model.labels_)
    noise = cluster_labels == -1
    cluster_ids, color_values = np.unique(cluster_labels[~noise], return_inverse=True)
    n_clusters = len(cluster_ids)

    if n_clusters:
        scatter = ax.scatter(
            embedding[~noise, 0],
            embedding[~noise, 1],
            c=color_values,
            cmap=plt.get_cmap("turbo", n_clusters),
            vmin=-0.5,
            vmax=n_clusters - 0.5,
            s=12,
            alpha=0.75,
        )

        colorbar = ax.figure.colorbar(scatter, ax=ax)
        step = max(1, int(np.ceil(n_clusters / 20)))
        ticks = np.arange(0, n_clusters, step)
        colorbar.set_ticks(ticks)
        colorbar.set_ticklabels([str(cluster_ids[t]) for t in ticks])
        colorbar.set_label("Cluster")

        members = embedding[~noise]
        centroids = np.array(
            [members[color_values == k].mean(axis=0) for k in range(n_clusters)]
        )
        ax.scatter(
            centroids[:, 0],
            centroids[:, 1],
            c="white",
            marker="*",
            s=300,
            edgecolors="black",
            linewidths=1.2,
            zorder=5,
            label="Centroid",
        )

    # Noise points in grey
    if noise.any():
        ax.scatter(
            embedding[noise, 0],
            embedding[noise, 1],
            c="gray",
            s=12,
            alpha=0.75,
            label="Noise (-1)",
        )

    if n_clusters or noise.any():
        ax.legend(loc="best")

    ax.set(
        xlabel="UMAP 1",
        ylabel="UMAP 2",
        title=title or "Clusters in UMAP space",
    )


def make_cluster_plot_panel(models, embedding, labels=None):
    models = list(models)
    if labels is None:
        labels = [None] * len(models)

    columns = min(2, len(models))
    rows = (len(models) + columns - 1) // columns
    fig, axes = plt.subplots(
        rows, columns, figsize=(8 * columns, 5 * rows), squeeze=False
    )

    for model, label, ax in zip(models, labels, axes.flat):
        make_cluster_plot(model, embedding, ax, label)

    for ax in axes.flat[len(models):]:
        ax.remove()

    fig.suptitle("Clustering assignments visualized with UMAP")
    fig.tight_layout()
    return fig, axes


def compare_clustering_parameter(X, estimator, parameter, values, embedding, return_time=False, **fixed_params):
    base_model = estimator(**fixed_params)
    values = list(values)

    models = []
    fit_times = []

    for value in values:
        model = clone(base_model).set_params(**{parameter: value})
        
        start_time = time.perf_counter()
        model.fit(X)
        end_time = time.perf_counter()
        
        models.append(model)
        fit_times.append(end_time - start_time)

    labels = [
        f"{type(base_model).__name__}: {parameter}={value}"
        for value in values
    ]

    # Return the times list if return_time is True
    if return_time:
        return make_cluster_plot_panel(models, embedding, labels), [l for l in labels], [m for m in models], fit_times

    return make_cluster_plot_panel(models, embedding, labels), [l for l in labels], [m for m in models]

In [ ]:
def get_centers(X, labels):
    values = X.to_numpy()
    ids = np.unique(labels[labels != -1])
    return np.array([values[labels == i].mean(axis=0) for i in ids])


def plot_anomalies(scores, outliers, deviation, X, embedding, title, threshold=None):
    ratio = deviation[outliers].mean(axis=0) / deviation[~outliers].mean(axis=0)
    ratio = pd.Series(ratio, index=X.columns)
    ratio = ratio[continuous_columns].sort_values(ascending=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(embedding[~outliers, 0], embedding[~outliers, 1], c="lightgray", s=6, label="Normal")
    axes[0].scatter(embedding[outliers, 0], embedding[outliers, 1], c="red", s=25, label="Outliers")
    axes[0].set(xlabel="UMAP 1", ylabel="UMAP 2", title="Outliers in UMAP space")
    axes[0].legend()

    bins = np.linspace(scores.min(), scores.max(), 61)
    axes[1].hist(scores, bins=bins)
    if threshold is None:
        axes[1].hist(scores[outliers], bins=bins, color="red", label="Noise")
    else:
        axes[1].axvline(threshold, color="red", linestyle="--", label="Threshold")
    axes[1].set_yscale("log")
    axes[1].set(xlabel="Distance to nearest centroid", ylabel="Points", title="Distance to centroid")
    axes[1].legend()

    axes[2].barh(ratio.index[::-1], ratio.values[::-1])
    axes[2].set(xlabel="Outlier / normal deviation", title="Features pushing outliers away")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

    print(ratio.round(2).to_string())
    print(f"Outliers Identified: {outliers.sum()} (out of {len(X)} total points)")
    if threshold is not None:
        print(f"Distance Threshold: {threshold:.2f}")
    print(X[outliers].head(10).round(2).to_string())

    return outliers


def analyze_kmeans_anomalies(kmeans_model, X, embedding, percentile=98):
    distances = kmeans_model.transform(X)
    scores = distances.min(axis=1)
    closest = distances.argmin(axis=1)

    threshold = np.percentile(scores, percentile)
    outliers = scores > threshold

    deviation = (X.to_numpy() - kmeans_model.cluster_centers_[closest]) ** 2
    title = f"K-Means Anomalies (Top {100 - percentile}%)"
    return plot_anomalies(scores, outliers, deviation, X, embedding, title, threshold)


def analyze_agglomerative_anomalies(agg_model, X, embedding, percentile=98):
    centers = get_centers(X, agg_model.labels_)
    deviation = (X.to_numpy() - centers[agg_model.labels_]) ** 2
    scores = np.sqrt(deviation.sum(axis=1))

    threshold = np.percentile(scores, percentile)
    outliers = scores > threshold

    title = f"Agglomerative ({agg_model.linkage}) Anomalies (Top {100 - percentile}%)"
    return plot_anomalies(scores, outliers, deviation, X, embedding, title, threshold)


def analyze_dbscan_anomalies(dbscan_model, X, embedding):
    outliers = dbscan_model.labels_ == -1
    if not outliers.any():
        print("DBSCAN found no noise points")
        return outliers

    centers = get_centers(X, dbscan_model.labels_)
    distances = pairwise_distances(X, centers)
    scores = distances.min(axis=1)
    closest = distances.argmin(axis=1)

    deviation = (X.to_numpy() - centers[closest]) ** 2
    title = f"DBSCAN Anomalies (noise points, eps={dbscan_model.eps}, min_samples={dbscan_model.min_samples})"
    return plot_anomalies(scores, outliers, deviation, X, embedding, title)

def analyze_centroids(model, X):
    if hasattr(model, "cluster_centers_"):
        centers = model.cluster_centers_
    else:
        centers = get_centers(X, model.labels_)

    centroids = pd.DataFrame(centers, columns=X.columns)
    centroids.index.name = 'Cluster'
    print("Cluster Centers (Average Feature Values):")
    print(centroids)
    print("\nFeature Variance Across Centroids:")
    print(centroids.var().sort_values(ascending=False))

### UMAP embedding
Fit once here and reused by every cluster plot below.

In [ ]:
embedding = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
).fit_transform(X)

### K-means Clustering

In [ ]:
k_values = range(1, 30)
sse = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
    sse.append(model.inertia_)

plt.figure(figsize=(10, 4))
plt.plot(k_values, sse, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("SSE (inertia)")
plt.title("K-means elbow plot")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

In [ ]:
cluster_plot_fig, labels, models = compare_clustering_parameter(
    X,
    KMeans,
    parameter="n_clusters",
    values=[2, 5, 8, 10],
    embedding=embedding,
    random_state = 0
)
plt.show()

In [ ]:
def get_results(labels, models):
    stats = []
    for label, model in zip(labels, models):
        pct_dict = (
            (pd.Series(model.labels_).value_counts(normalize=True) * 100)
            .round(1)
            .to_dict()
        )

        stats.append(
            {
                "Model": label,
                "Iterations": model.n_iter_,
                "% per Cluster": pct_dict,
                "SSE": model.inertia_
            }
        )

    return pd.DataFrame(stats)

results_df = get_results(labels, models)
print(results_df.to_string(index=False))

In [ ]:
label = labels[3]
model = models[3]

print(label)
outliers = analyze_kmeans_anomalies(model, X, embedding)
analyze_centroids(model, X)

In [ ]:
cluster_plot_fig, labels, models= compare_clustering_parameter(
    X,
    KMeans,
    parameter="n_init",
    values=[1, 5, 10, 20],
    embedding=embedding,
    random_state = 0
)
plt.show()

In [ ]:
results_df = get_results(labels, models)
print(results_df.to_string(index=False))

In [ ]:
print(labels[3])
outliers = analyze_kmeans_anomalies(models[3], X, embedding)

In [ ]:
cluster_plot_fig, labels, models= compare_clustering_parameter(
    X,
    KMeans,
    parameter="max_iter",
    values=[1, 10],
    embedding=embedding,
    random_state = 0
)
plt.show()

In [ ]:
results_df = get_results(labels, models)
print(results_df.to_string(index=False))

## Agglomerative Clustering

In [ ]:
def analyze_kmeans_anomalies(kmeans_model, X, embedding, percentile=98):
    distances = kmeans_model.transform(X)
    scores = distances.min(axis=1)
    closest = distances.argmin(axis=1)

    threshold = np.percentile(scores, percentile)
    outliers = scores > threshold

    deviation = (X.to_numpy() - kmeans_model.cluster_centers_[closest]) ** 2
    ratio = deviation[outliers].mean(axis=0) / deviation[~outliers].mean(axis=0)
    ratio = pd.Series(ratio, index=X.columns)
    ratio = ratio[continuous_columns].sort_values(ascending=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(embedding[~outliers, 0], embedding[~outliers, 1], c="lightgray", s=6, label="Normal")
    axes[0].scatter(embedding[outliers, 0], embedding[outliers, 1], c="red", s=25, label="Outliers")
    axes[0].set(xlabel="UMAP 1", ylabel="UMAP 2", title="Outliers in UMAP space")
    axes[0].legend()

    axes[1].hist(scores, bins=60)
    axes[1].axvline(threshold, color="red", linestyle="--", label="Threshold")
    axes[1].set_yscale("log")
    axes[1].set(xlabel="Distance to nearest centroid", ylabel="Points", title="Distance to centroid")
    axes[1].legend()

    axes[2].barh(ratio.index[::-1], ratio.values[::-1])
    axes[2].set(xlabel="Outlier / normal deviation", title="Features pushing outliers away")

    fig.suptitle(f"K-Means Anomalies (Top {100 - percentile}%)")
    plt.tight_layout()
    plt.show()

    print(ratio.round(2).to_string())
    print(f"Outliers Identified: {outliers.sum()} (out of {len(X)} total points)")
    print(f"Distance Threshold: {threshold:.2f}")
    print(X[outliers].head(10).round(2).to_string())

    return outliers

In [ ]:
connectivity = kneighbors_graph(X, n_neighbors=100, include_self=False)

cluster_plot_fig, labels, models, fit_times = compare_clustering_parameter(
        X,
        AgglomerativeClustering,
        parameter="linkage",
        values=['ward', 'average', 'complete', 'single'],
        return_time = True,
        embedding=embedding,
        connectivity = connectivity,
        n_clusters=8
    )
plt.show()

In [ ]:
def get_results(labels, models, fit_times, X):
    stats = []
    for label, model, fit_time in zip(labels, models, fit_times):
        pct_dict = (
            (pd.Series(model.labels_).value_counts(normalize=True) * 100)
            .round(1)
            .to_dict()
        )

        n_unique_labels = len(np.unique(model.labels_))
        
        sil_score = silhouette_score(X, model.labels_)
        sil_score = round(sil_score, 4)

        stats.append(
            {
                "Model": label,
                "# of Clusters": n_unique_labels,
                "% per Cluster": pct_dict,
                "Silhouette Score": sil_score,
                "Fit Time": fit_time
            }
        )

    return pd.DataFrame(stats)

results_df = get_results(labels, models, fit_times, X)
print(results_df.to_string(index=False))

In [ ]:
print(labels[1])
outliers = analyze_agglomerative_anomalies(models[1], X, embedding)
analyze_centroids(models[1], X)

## DBSCAN

In [ ]:
min_samples_values = [10, 50, 100, 200]
min_samples_values = [m for m in min_samples_values if m < len(X)]

fig, ax = plt.subplots(figsize=(8, 5))
all_k_distances = []

for min_samples in min_samples_values:
    distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X).kneighbors(X)
    k_distances = np.sort(distances[:, -1])
    all_k_distances.extend(k_distances)
    ax.plot(k_distances, label=f"min_samples={min_samples}")

ax.set(
    xlabel="Points sorted by distance",
    ylabel="Distance to k-th neighbor",
    title="DBSCAN k-distance curves",
)
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
cluster_plot_fig, labels, models, times = compare_clustering_parameter(
    X,
    DBSCAN,
    parameter="algorithm",
    return_time = True,
    values=['auto', 'ball_tree', 'kd_tree', 'brute'],
    embedding=embedding,
)
plt.show()

In [ ]:

results_df = get_results(labels, models, times, X)
print(results_df.to_string(index=False))

In [ ]:
cluster_plot_fig, labels, models, times= compare_clustering_parameter(
    X,
    DBSCAN,
    return_time = True,
    parameter="eps",
    values=[0.2, 0.3, 0.4, 0.5],
    min_samples=5,
    embedding=embedding,
)
plt.show()

In [ ]:
results_df = get_results(labels, models, times, X)
print(results_df.to_string(index=False))

In [ ]:
cluster_plot_fig, labels, models, times = compare_clustering_parameter(
    X,
    DBSCAN,
    parameter="min_samples",
    return_time=True,
    values=[10, 50, 200, 1000],
    embedding=embedding,
)
plt.show()

In [ ]:

results_df = get_results(labels, models, times, X)
print(results_df.to_string(index=False))

In [ ]:
print(labels[2])
outliers = analyze_dbscan_anomalies(models[2], X, embedding)
analyze_centroids(models[2], X)

## Selected models

In [ ]:
selected_kmeans = KMeans(n_clusters=4, random_state=0, max_iter=10).fit(X)

In [ ]:
connectivity = kneighbors_graph(X, n_neighbors=100, include_self=False)
selected_agglomerative = AgglomerativeClustering(linkage='ward', connectivity=connectivity, n_clusters=4).fit(X)

In [ ]:
selected_dbscan = DBSCAN(eps=0.4, min_samples=200).fit(X)

In [ ]:
# Extra clustering for a "same method, different parameters" relative comparison
kmeans_seed1 = KMeans(n_clusters=4, random_state=1, max_iter=10).fit(X)

### Clustering analysis

In [ ]:

labels = {
    'KMeans': selected_kmeans.labels_,
    'Agglomerative (Ward)': selected_agglomerative.labels_,
    'DBSCAN': selected_dbscan.labels_,
}


def non_noise(lbls):
    """Boolean mask of points not labelled as DBSCAN noise (-1)."""
    return lbls != -1

### Clustering vis

In [ ]:
make_cluster_plot_panel(
    [selected_kmeans, selected_agglomerative, selected_dbscan, kmeans_seed1],
    embedding,
    labels=[
        'KMeans (n_clusters=4, random_state=0)',
        'Agglomerative (Ward)',
        'DBSCAN (eps=0.4, min_samples=200)'
    ]
)

### Internal Indices

In [ ]:
def sse(X, lbls):
    """Sum of squared distances of each point to its cluster centroid.
    Noise points (-1) are excluded."""
    mask = non_noise(lbls)
    Xm, lm = X[mask], lbls[mask]
    total = 0.0
    for c in np.unique(lm):
        pts = Xm[lm == c]
        total += ((pts - pts.mean(axis=0)) ** 2).to_numpy().sum()
    return total


def silhouette(X, lbls, sample_size=10000, seed=0):
    mask = non_noise(lbls)
    Xm, lm = X[mask], lbls[mask]
    if len(np.unique(lm)) < 2:
        return np.nan
    ss = min(sample_size, len(Xm)) if len(Xm) > sample_size else None
    return metrics.silhouette_score(Xm, lm, sample_size=ss, random_state=seed)


def proximity_incidence_correlation(X, lbls, name, max_points=3000, seed=0, plot=True):
    """Correlation between the distance matrix and the (ideal) incidence matrix.
    Points are sorted by cluster label so that clusters appear as blocks.
    A good clustering gives a strongly NEGATIVE correlation (same cluster -> small distance)."""
    mask = non_noise(lbls)
    Xm, lm = X[mask], lbls[mask]

    rng = np.random.RandomState(seed)
    if len(Xm) > max_points:                      # subsample: n x n matrices get big
        idx = rng.choice(len(Xm), max_points, replace=False)
        Xm, lm = Xm.iloc[idx], lm[idx]

    order = np.argsort(lm, kind='stable')
    Xm, lm = Xm.iloc[order], lm[order]

    dist = pairwise_distances(Xm)
    incidence = (lm[:, None] == lm[None, :]).astype(float)

    iu = np.triu_indices_from(dist, k=1)           # upper triangle, no diagonal
    corr = np.corrcoef(dist[iu], incidence[iu])[0, 1]

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        sns.heatmap(dist, ax=axes[0], cmap='viridis', xticklabels=False, yticklabels=False)
        axes[0].set_title(f'{name}: distance matrix (sorted by cluster)')
        sns.heatmap(incidence, ax=axes[1], cmap='binary', xticklabels=False, yticklabels=False)
        axes[1].set_title(f'{name}: incidence matrix\ncorr = {corr:.3f}')
        plt.tight_layout()
        plt.show()
    return corr


internal_rows = []
for name, lbls in labels.items():
    internal_rows.append({
        'Method': name,
        'n_clusters': len(set(lbls) - {-1}),
        'n_noise': int((lbls == -1).sum()),
        'SSE': sse(X, lbls),
        'Silhouette': silhouette(X, lbls),
        'Dist-Incidence corr': proximity_incidence_correlation(X, lbls, name),
    })
internal_df = pd.DataFrame(internal_rows).set_index('Method')
print('=== (2) Internal indices ===')
internal_df.round(4)

### Relative Indices
Chosen pairs
 - KMeans(seed 0) vs KMeans(seed 1): stability of k-means w.r.t. initialisation
 - KMeans vs Agglomerative(Ward): both minimise variance -> expect similar partitions
 - KMeans vs DBSCAN and Agglomerative vs DBSCAN: centroid/variance-based vs density-based

In [ ]:
pairs = [
    ('KMeans (seed 0)', selected_kmeans.labels_, 'KMeans (seed 1)', kmeans_seed1.labels_),
    ('KMeans', labels['KMeans'], 'Agglomerative (Ward)', labels['Agglomerative (Ward)']),
    ('KMeans', labels['KMeans'], 'DBSCAN', labels['DBSCAN']),
    ('Agglomerative (Ward)', labels['Agglomerative (Ward)'], 'DBSCAN', labels['DBSCAN']),
]

relative_rows = []
for n1, l1, n2, l2 in pairs:
    relative_rows.append({
        'Clustering A': n1,
        'Clustering B': n2,
        'SSE A': sse(X, l1),
        'SSE B': sse(X, l2),
        'Adjusted Rand': metrics.adjusted_rand_score(l1, l2),
        'NMI': metrics.normalized_mutual_info_score(l1, l2),
        'AMI': metrics.adjusted_mutual_info_score(l1, l2),
    })
relative_df = pd.DataFrame(relative_rows)
print('\n=== (3) Relative indices ===')
relative_df.round(4)

### (4) EXTERNAL INDICES  (clusterings vs. Severity)

In [ ]:

external_rows = []
contingency = {}
for name, lbls in labels.items():
    external_rows.append({
        'Method': name,
        'Homogeneity': metrics.homogeneity_score(y, lbls),
        'Completeness': metrics.completeness_score(y, lbls),
        'V-measure': metrics.v_measure_score(y, lbls),
    })
    cm = contingency_matrix(y, lbls)
    cols = [f'cluster {c}' if c != -1 else 'noise' for c in np.unique(lbls)]
    contingency[name] = pd.DataFrame(cm, index=[f'Severity {s}' for s in np.unique(y)], columns=cols)

external_df = pd.DataFrame(external_rows).set_index('Method')
print('\n=== (4) External indices ===')
external_df.round(4)

In [ ]:

fig, axes = plt.subplots(1, len(contingency), figsize=(6 * len(contingency) + 15, 6))
for ax, (name, cdf) in zip(np.atleast_1d(axes), contingency.items()):
    sns.heatmap(cdf, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'Contingency matrix: {name}')
plt.tight_layout()
plt.show()

### Other quantitative results across methods

In [ ]:

sizes_df = pd.DataFrame({
    name: pd.Series(pd.Series(lbls).value_counts().sort_index())
    for name, lbls in labels.items()
}).fillna(0).astype(int)
sizes_df.index.name = 'cluster label (-1 = noise)'
print('\n=== (5) Cluster sizes ===')
sizes_df


In [ ]:
summary_df = internal_df.join(external_df)
print('\n=== (5) Summary across methods ===')
summary_df.round(4)

In [ ]:
score_cols = ['Silhouette', 'Homogeneity', 'Completeness', 'V-measure']
summary_df[score_cols].plot(kind='bar', figsize=(9, 4))
plt.title('Evaluation scores by clustering method')
plt.ylabel('score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

names = list(labels.keys())
ari = pd.DataFrame(
    [[metrics.adjusted_rand_score(labels[a], labels[b]) for b in names] for a in names],
    index=names, columns=names)
plt.figure(figsize=(5, 4))
sns.heatmap(ari, annot=True, fmt='.3f', cmap='coolwarm', vmin=0, vmax=1)
plt.title('Adjusted Rand index between clusterings')
plt.tight_layout()
plt.show()

## Qualitative Analysis and Visualizations


In [ ]:
from sklearn.manifold import MDS, TSNE

viz_n = min(1000, len(X))
viz_idx = np.random.RandomState(42).choice(len(X), size=viz_n, replace=False)
X_viz = X.iloc[viz_idx]

viz_embeddings = {
    "UMAP": embedding[viz_idx],
    "t-SNE": TSNE(
        n_components=2,
        perplexity=min(30, max(1, (viz_n - 1) / 3)),
        init="pca",
        learning_rate="auto",
        random_state=42,
    ).fit_transform(X_viz),
    "MDS": MDS(
        n_components=2,
        init="random",
        random_state=42,
        n_init=1,
        max_iter=60,
    ).fit_transform(X_viz),
}

selected_labels = {
    "K-Means": selected_kmeans.labels_,
    "Agglomerative (Ward)": selected_agglomerative.labels_,
    "DBSCAN": selected_dbscan.labels_,
}

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
for row, (method, cluster_labels) in enumerate(selected_labels.items()):
    sample_labels = cluster_labels[viz_idx]
    noise = sample_labels == -1
    cluster_ids, color_values = np.unique(sample_labels[~noise], return_inverse=True)

    for column, (projection_name, coordinates) in enumerate(viz_embeddings.items()):
        ax = axes[row, column]
        if len(cluster_ids):
            ax.scatter(
                coordinates[~noise, 0],
                coordinates[~noise, 1],
                c=color_values,
                cmap=plt.get_cmap("turbo", len(cluster_ids)),
                s=9,
                alpha=0.7,
                rasterized=True,
            )
        if noise.any():
            ax.scatter(
                coordinates[noise, 0],
                coordinates[noise, 1],
                c="gray",
                s=9,
                alpha=0.7,
                rasterized=True,
            )
        if row == 0:
            ax.set_title(projection_name)
        if column == 0:
            ax.set_ylabel(method)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Selected clusterings across UMAP, t-SNE, and MDS", y=0.99)
fig.text(
    0.5,
    0.015,
    "Colors denote cluster assignments within each panel; gray points are DBSCAN noise. "
    f"Same {viz_n:,}-accident subset in every projection.",
    ha="center",
)
fig.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.show()

In [ ]:
domain_features = [column for column in continuous_columns if column in X.columns]
domain_values = pd.DataFrame(
    scaler.inverse_transform(X[continuous_columns]),
    columns=continuous_columns,
    index=X.index,
)
domain_values["Severity"] = y.to_numpy()

cluster_profile_tables = []
severity_mix_tables = []
representative_rows = []
centroid_separation_rows = []

for method, cluster_labels in selected_labels.items():
    analysis_frame = domain_values.copy()
    analysis_frame["Cluster"] = cluster_labels
    assigned = analysis_frame[analysis_frame["Cluster"] != -1]
    grouped = assigned.groupby("Cluster", sort=True)

    profile = grouped[domain_features + ["Severity"]].agg(["median", "mean"])
    profile.columns = [f"{feature}_{stat}" for feature, stat in profile.columns]
    profile.insert(0, "n", grouped.size())
    profile.insert(0, "Method", method)
    profile.insert(1, "Cluster", profile.index)
    cluster_profile_tables.append(profile.reset_index(drop=True))

    severity_mix = pd.crosstab(
        assigned["Cluster"], assigned["Severity"], normalize="index"
    ).add_prefix("Severity_")
    severity_mix.insert(0, "Method", method)
    severity_mix.insert(1, "Cluster", severity_mix.index)
    severity_mix_tables.append(severity_mix.reset_index(drop=True))

    cluster_ids = np.unique(cluster_labels[cluster_labels != -1])
    centers = np.vstack([
        X.iloc[np.flatnonzero(cluster_labels == cluster_id)].mean(axis=0).to_numpy()
        for cluster_id in cluster_ids
    ])

    center_distances = pairwise_distances(centers)
    upper_i, upper_j = np.triu_indices_from(center_distances, k=1)
    closest_pair = np.argmin(center_distances[upper_i, upper_j])
    farthest_pair = np.argmax(center_distances[upper_i, upper_j])
    centroid_separation_rows.append({
        "Method": method,
        "Closest cluster pair": (
            int(cluster_ids[upper_i[closest_pair]]),
            int(cluster_ids[upper_j[closest_pair]]),
        ),
        "Closest center distance": center_distances[upper_i[closest_pair], upper_j[closest_pair]],
        "Farthest cluster pair": (
            int(cluster_ids[upper_i[farthest_pair]]),
            int(cluster_ids[upper_j[farthest_pair]]),
        ),
        "Farthest center distance": center_distances[upper_i[farthest_pair], upper_j[farthest_pair]],
    })

    for cluster_id, center in zip(cluster_ids, centers):
        member_positions = np.flatnonzero(cluster_labels == cluster_id)
        member_values = X.iloc[member_positions].to_numpy()
        distances_to_center = np.linalg.norm(member_values - center, axis=1)
        typical_positions = member_positions[np.argsort(distances_to_center)[:2]]
        for position in typical_positions:
            row = {
                "Method": method,
                "Cluster": int(cluster_id),
                "Row index": X.index[position],
                "Distance to cluster center": np.linalg.norm(
                    X.iloc[position].to_numpy() - center
                ),
                "Severity": y.iloc[position],
            }
            row.update(domain_values.iloc[position][domain_features].to_dict())
            representative_rows.append(row)

cluster_profile_df = pd.concat(cluster_profile_tables, ignore_index=True).set_index(
    ["Method", "Cluster"]
)
severity_mix_df = pd.concat(severity_mix_tables, ignore_index=True).set_index(
    ["Method", "Cluster"]
)
representative_df = pd.DataFrame(representative_rows).set_index(
    ["Method", "Cluster", "Row index"]
)
centroid_separation_df = pd.DataFrame(centroid_separation_rows).set_index("Method")

print("Cluster profiles: median and mean in original accident units")
display(cluster_profile_df.round(2))
print("\nSeverity proportions within each cluster")
display(severity_mix_df.round(3))
print("\nTwo actual accidents nearest each cluster's feature-space center")
display(representative_df.round(2))
print("\nMost similar and most separated cluster centers in scaled feature space")
display(centroid_separation_df.round(3))

### Cross-method outlier inspection

In [ ]:
kmeans_scores = selected_kmeans.transform(X).min(axis=1)
kmeans_outliers = kmeans_scores > np.percentile(kmeans_scores, 98)

agg_labels = selected_agglomerative.labels_
agg_cluster_ids = np.unique(agg_labels)
agg_centers = {
    cluster_id: X.iloc[np.flatnonzero(agg_labels == cluster_id)].mean(axis=0).to_numpy()
    for cluster_id in agg_cluster_ids
}
agg_center_by_row = np.vstack([agg_centers[cluster_id] for cluster_id in agg_labels])
agg_scores = np.linalg.norm(X.to_numpy() - agg_center_by_row, axis=1)
agg_outliers = agg_scores > np.percentile(agg_scores, 98)

dbscan_noise = selected_dbscan.labels_ == -1
outlier_masks = {
    "K-Means top 2% distance": kmeans_outliers,
    "Agglomerative top 2% distance": agg_outliers,
    "DBSCAN noise": dbscan_noise,
}

outlier_overlap_rows = []
for (method_a, mask_a), (method_b, mask_b) in itertools.combinations(
    outlier_masks.items(), 2
):
    intersection = int(np.logical_and(mask_a, mask_b).sum())
    union = int(np.logical_or(mask_a, mask_b).sum())
    outlier_overlap_rows.append({
        "Approach A": method_a,
        "Approach B": method_b,
        "Shared accidents": intersection,
        "Jaccard overlap": intersection / union if union else np.nan,
    })

outlier_votes = np.sum(np.column_stack(list(outlier_masks.values())), axis=1)
outlier_groups = {
    **outlier_masks,
    "Flagged by 2+ approaches": outlier_votes >= 2,
    "Flagged by all 3 approaches": outlier_votes == len(outlier_masks),
}

outlier_domain_rows = []
for group_name, mask in outlier_groups.items():
    flagged = domain_values.loc[mask]
    unflagged = domain_values.loc[~mask]
    row = {"Group": group_name, "n": int(mask.sum())}
    for feature in domain_features + ["Severity"]:
        row[f"{feature} median (flagged)"] = flagged[feature].median()
        row[f"{feature} median (unflagged)"] = unflagged[feature].median()
    for severity_value in np.sort(y.unique()):
        row[f"Severity {severity_value} proportion"] = (
            float((y.to_numpy()[mask] == severity_value).mean()) if mask.any() else np.nan
        )
    outlier_domain_rows.append(row)

outlier_overlap_df = pd.DataFrame(outlier_overlap_rows)
outlier_domain_df = pd.DataFrame(outlier_domain_rows).set_index("Group")

print("Pairwise overlap between outlier definitions")
display(outlier_overlap_df.round(3))
print("\nDomain characteristics of flagged cases versus each group's unflagged cases")
display(outlier_domain_df.round(3))

shared_positions = np.flatnonzero(outlier_votes >= 2)[:10]
shared_outlier_examples = []
for position in shared_positions:
    row = {
        "Row index": X.index[position],
        "Methods flagged": ", ".join(
            method for method, mask in outlier_masks.items() if mask[position]
        ),
        "Severity": y.iloc[position],
    }
    row.update(domain_values.iloc[position][domain_features].to_dict())
    shared_outlier_examples.append(row)

print("\nExample accidents flagged by at least two approaches (up to 10)")
if shared_outlier_examples:
    display(pd.DataFrame(shared_outlier_examples).set_index("Row index").round(2))
else:
    print("No accidents were flagged by two or more approaches.")